## 냉장고 테이블 재구성

In [1]:
from pathlib import Path
import pandas as pd
import re

# =========================
# 경로 설정
# =========================
input_path = Path(r"C:/project_skn/project4/4th_project/products/data/preprocessing/refrigerator/lge_ref_all_products.csv")
output_path = Path("REF.csv")

# =========================
# 변환 함수
# =========================
def blank_if_na(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def clean_int(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).replace(",", "")
    nums = re.findall(r"\d+", text)

    if not nums:
        return pd.NA

    return int(nums[0])


def clean_float(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).replace(",", "")

    # 괄호 뒤 설명 제거: 3,500 (세탁 : 2,100 , 건조 : 1,400) -> 3500
    text = text.split("(")[0]

    nums = re.findall(r"\d+(?:\.\d+)?", text)

    if not nums:
        return pd.NA

    return float(nums[0])


def clean_proficiency(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).strip()

    if "대상 외" in text or "비대상" in text:
        return pd.NA

    # 세탁기 2등급 / 건조기 1등급 -> 세탁기 기준
    washer_match = re.search(r"세탁기\s*(\d+)", text)
    if washer_match:
        return int(washer_match.group(1))

    nums = re.findall(r"\d+", text)
    if not nums:
        return pd.NA

    return int(nums[0])


def split_size(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA, pd.NA, pd.NA

    text = str(value).replace(",", "").replace('"', "").strip()
    parts = re.split(r"\s*[xX×]\s*", text)

    if len(parts) < 3:
        return pd.NA, pd.NA, pd.NA

    width = clean_int(parts[0])
    height = clean_int(parts[1])
    depth = clean_int(parts[2])

    return width, height, depth


def clean_weight(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA

    text = str(value).replace(",", "").strip()

    # 전체 162 / 세탁 91 / 건조 71 -> 전체 값
    total_match = re.search(r"전체\s*(\d+(?:\.\d+)?)", text)
    if total_match:
        return int(float(total_match.group(1)))

    nums = re.findall(r"\d+(?:\.\d+)?", text)
    if not nums:
        return pd.NA

    return int(float(nums[0]))


def clean_binary_yes_no(value):
    if pd.isna(value) or str(value).strip() == "":
        return 0

    text = str(value).strip()

    if "있음" in text:
        return 1

    if "없음" in text or "제공안함" in text:
        return 0

    return 0


def clean_ice_maker(value):
    if pd.isna(value) or str(value).strip() == "":
        return 0

    text = str(value).strip()

    no_keywords = ["제공안함", "없음", "미지원", "해당없음", "X", "x"]
    yes_keywords = ["아이스", "트레이", "메이커", "오토", "트위스트"]

    if any(keyword in text for keyword in no_keywords):
        return 0

    if any(keyword in text for keyword in yes_keywords):
        return 1

    return 0


# =========================
# 원본 읽기
# =========================
df = pd.read_csv(input_path)

# =========================
# 제품 크기 분리
# =========================
size_values = df["제품 크기(mm)"].apply(split_size)

df["width(mm)"] = size_values.apply(lambda x: x[0])
df["height(mm)"] = size_values.apply(lambda x: x[1])
df["depth(mm)"] = size_values.apply(lambda x: x[2])

# =========================
# 새 DataFrame 구성
# =========================
new_df = pd.DataFrame({
    "product_code": [f"REF{i:03d}" for i in range(len(df))],
    "name": df["제품명"].apply(blank_if_na),
    "img_link": df["이미지 링크"].apply(blank_if_na),
    "price": df["가격(원)"].apply(clean_int),
    "proficiency_level": df["에너지 소비효율등급"].apply(clean_proficiency),
    "power_consum(W)": df["소비전력(W)"].apply(clean_float),
    "install_type": df["설치타입"].apply(blank_if_na),
    "door_cnt": df["도어 개수"].apply(clean_int),
    "total_cap(L)": df["전체 용량(L)"].apply(clean_int),
    "refrige_cap(L)": df["냉장 용량(L)"].apply(clean_int),
    "freeze_cap(L)": df["냉동 용량(L)"].apply(clean_int),
    "width(mm)": df["width(mm)"],
    "height(mm)": df["height(mm)"],
    "depth(mm)": df["depth(mm)"],
    "weight(kg)": df["무게(kg)"].apply(clean_weight),
    "color": df["색상"].apply(blank_if_na),
    "door_type": df["도어 재질"].apply(blank_if_na),
    "cool_type": df["냉각방식"].apply(blank_if_na),
    "smart_diag": df["스마트 진단"].apply(clean_binary_yes_no),
    "ice_maker": df["아이스메이커"].apply(clean_ice_maker),
    "manual_link": df["제품 사용 설명서"].apply(blank_if_na)
})

# =========================
# 타입 지정
# =========================
str_cols = [
    "product_code", "name", "img_link",
    "install_type", "color", "door_type",
    "cool_type", "manual_link"
]

int_cols = [
    "price", "proficiency_level", "door_cnt",
    "total_cap(L)", "refrige_cap(L)", "freeze_cap(L)",
    "width(mm)", "height(mm)", "depth(mm)",
    "weight(kg)", "smart_diag", "ice_maker"
]

for col in str_cols:
    new_df[col] = new_df[col].astype("string")

for col in int_cols:
    new_df[col] = new_df[col].astype("Int64")

new_df["power_consum(W)"] = new_df["power_consum(W)"].astype("Float64")

# =========================
# 저장
# =========================
new_df.to_csv(output_path, index=False, encoding="utf-8-sig", na_rep="")

print(new_df.head())
print(f"저장 완료: {output_path}")

  product_code                                               name  \
0       REF000  LG 트롬 오브제컬렉션 워시타워 + LG 디오스 AI 오브제컬렉션 냉장고 (매직스페이스)   
1       REF001  LG 트롬 오브제컬렉션 워시타워 + LG 디오스 AI 오브제컬렉션 냉장고 (매직스페이스)   
2       REF002                                             LG 냉동고   
3       REF003                                           LG 일반냉장고   
4       REF004                                           LG 일반냉장고   

                                            img_link    price  \
0  https://www.lge.co.kr/kr/images/wash-tower/md1...  6280000   
1  https://www.lge.co.kr/kr/images/wash-tower/md1...  5780000   
2  https://www.lge.co.kr/kr/images/refrigerators/...   700000   
3  https://www.lge.co.kr/kr/images/refrigerators/...   260000   
4  https://www.lge.co.kr/kr/images/refrigerators/...   320000   

   proficiency_level  power_consum(W) install_type  door_cnt  total_cap(L)  \
0                  2           3500.0        프리스탠딩         2           832   
1                  2           3050.0 